# Marjum 2026-07 — Joint MCMC: Visual Review (Summary)

A **diagnostic artifact, not a decision**. The joint run over the transmitter,
the antenna and all 29 constraining camera poses **did not converge**: 212 of
213 coordinates miss both Rhat ≤ 1.01 and ESS ≥ 400. Nothing here should be
quoted as a posterior.

What this document is for is showing *what* is failing to mix and *where things
sit in space*, so the next step can be chosen with the evidence in hand.

**Reproducibility status (2026-09-21):** archived inputs were restored from
`limbo-reprocess:/mnt/data02/eigsep/terrain/` and verified against the frozen
archive and run-manifest SHA-256 hashes. This notebook now executes from the
`arp` kernel; the non-convergence warning above still governs interpretation.

Companion document: **per-camera review**, one page per camera — parameter
traces, that camera's own draw cloud, and its horizon spread.

In [ ]:
import json, warnings, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

warnings.simplefilter('ignore', RuntimeWarning)
def find_terrain_root(start=Path.cwd()):
    for parent in (start, *start.parents):
        candidate = parent / 'terrain'
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the sibling terrain repository')

WORKSPACE = find_terrain_root()
sys.path.insert(0, str(WORKSPACE))
RUN = WORKSPACE / 'joint_posterior_v1'

required = [RUN / 'manifest.json', RUN / 'convergence.json', RUN / 'combined.npz']
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing archived MCMC inputs:\n' +
                            '\n'.join(f'  - {path}' for path in missing))

manifest = json.loads((RUN / 'manifest.json').read_text())
conv = json.loads((RUN / 'convergence.json').read_text())
with np.load(RUN / 'combined.npz', allow_pickle=True) as z:
    draws = np.asarray(z['draws'], float)
    rhat = np.asarray(z['rhat'], float)
    ess = np.asarray(z['ess_bulk'], float)
    names = [str(s) for s in z['names']]
    keys = [str(s) for s in z['keys']]
nchain, ndraw, nparam = draws.shape
PRM = ('e', 'n', 'u', 'th', 'ph', 'ti', 'logf')
flat = draws.reshape(-1, nparam)
HR = WORKSPACE / 'horizon_range'
hr_summary = json.loads((HR / 'summary.json').read_text()) if (HR / 'summary.json').exists() else {}

print('chains', nchain, ' retained draws', ndraw, ' coordinates', nparam, ' cameras', len(keys))
print('state sampled :', manifest['state_file'])
g = manifest['geometry_release']
print('release pinned:', g['verified'], '--', g.get('reason', 'matches pin'))
print()
print(conv['verdict'])

## 1. What is stiff

Blocks ranked by median effective sample size. The pattern is the finding: the
**one block that is not a random-walk Metropolis update — the GPS bias, drawn
exactly by Gibbs from its conditional Gaussian — is the only one that mixes.**
Everything sampled by random walk sits two orders of magnitude below it. That
points at the kernel rather than at the model or the data.

In [ ]:
def block_of(n):
    if n.startswith('cam'):
        s = n.split('_')[1]
        return ('camera position' if s in ('e', 'n', 'u')
                else 'camera focal length' if s == 'logf' else 'camera angles')
    if n.startswith(('antenna_e', 'antenna_n', 'antenna_u')):
        return 'antenna position'
    if n.startswith(('transmitter_e', 'transmitter_n', 'transmitter_u')):
        return 'transmitter position'
    return 'label scatter' if 'extra' in n else 'GPS bias (exact Gibbs)'

from collections import defaultdict
g = defaultdict(list)
for n, r, e in zip(names, rhat, ess):
    g[block_of(n)].append((r, e))
print(f"{'block':28s}{'n':>4}{'med ESS':>10}{'min ESS':>9}{'med Rhat':>10}{'max Rhat':>10}")
print('-' * 71)
for k, v in sorted(g.items(), key=lambda kv: np.median([x[1] for x in kv[1]])):
    r = np.array([x[0] for x in v]); e = np.array([x[1] for x in v])
    print(f'{k:28s}{len(v):>4}{np.median(e):10.0f}{e.min():9.0f}{np.median(r):10.2f}{r.max():10.2f}')
print()
print('Thresholds: ESS >= 400, Rhat <= 1.01.  Nominal draws = %d x %d = %d.'
      % (nchain, ndraw, nchain * ndraw))

In [ ]:
def act(x, maxlag=3000):
    x = x - x.mean(); n = len(x)
    f = np.fft.rfft(x, 2 * n)
    ac = np.fft.irfft(f * np.conj(f))[:maxlag].real
    ac /= ac[0]
    s = 1.0
    for i in range(1, maxlag):
        if ac[i] <= 0:
            break
        s += 2 * ac[i]
    return s

print('Integrated autocorrelation time tau, chain 0 (%d draws):' % ndraw)
print(f"{'coordinate':24s}{'tau (draws)':>13}{'indep/chain':>13}")
for nm in ['cam2159_logf', 'cam2213_e', 'antenna_u', 'transmitter_u',
           'antenna_extra_px', 'transmitter_extra_px', 'gps_bias_e']:
    if nm in names:
        t = act(draws[0, :, names.index(nm)])
        print(f'{nm:24s}{t:13.0f}{ndraw / t:13.1f}')
print()
print('cam2159_logf is the worst coordinate in the run: ~4 independent samples')
print('per chain. The likely mechanism is the focal/depth degeneracy -- focal')
print('length trades against camera distance along a ridge a random walk crawls')
print('rather than crosses -- sharpest for 2159, which sits ~4.5 m from the')
print('transmitter with a long lens. This is a SAMPLING pathology; 2159\'s pose')
print('itself is correct on every independent check after the re-fit.')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].hist(rhat, bins=45, color='tab:blue')
axs[0].axvline(1.01, color='r', ls='--', label='threshold 1.01')
axs[0].set_xlabel('split-Rhat'); axs[0].set_ylabel('coordinates'); axs[0].legend(fontsize=7)
axs[0].set_title(f'Rhat, {nparam} coordinates', fontsize=9)
axs[1].hist(ess, bins=45, color='tab:green')
axs[1].axvline(400, color='r', ls='--', label='threshold 400')
axs[1].set_xlabel('bulk ESS'); axs[1].legend(fontsize=7)
axs[1].set_title('Effective sample size', fontsize=9)
plt.tight_layout(); plt.show()

## 1b. 2159 is not slow-mixing — its chains are in *separate modes*

The per-camera page for 2159 shows the eight chains sitting at visibly
different levels rather than exploring a common distribution, and its E/N draw
cloud splits into two clumps. That is a qualitatively different failure from
slow mixing, and the numbers below confirm it.

In [ ]:
b = keys.index('2159') * 7
print('2159 per-chain means:')
print(f"{'chain':>6}{'e':>10}{'n':>10}{'u':>10}{'focal':>10}")
for c in range(nchain):
    m = draws[c, :, b:b + 7].mean(0)
    print(f'{c:6d}{m[0]:10.2f}{m[1]:10.2f}{m[2]:10.2f}{np.exp(m[6]):10.0f}')
n = draws[:, :, b + 1]
print()
print('n: pooled sd %.2f m | between-chain sd of means %.2f m | within-chain sd %.2f m'
      % (n.std(), n.mean(1).std(), n.std(1).mean()))
print('between/within ratio %.1f -- a ratio >> 1 means the chains occupy separate regions'
      % (n.mean(1).std() / n.std(1).mean()))
print('sorted chain means:', np.round(np.sort(n.mean(1)), 2))
print()
print('Two clusters, ~4.8 m apart in northing:')
print('  chains 0,4,6,7 -> n ~ 2029.0-2029.6, u ~ 1685.2-1685.7, focal ~ 2893-2936 px')
print('  chains 1,2,3   -> n ~ 2033.7-2033.9, u ~ 1688.2-1688.6, focal ~ 3076-3084 px')
print('  chain 5        -> in between')
print()
print('Note the direction of the split: the further-back mode also has the LONGER')
print('focal length. That is the focal/depth degeneracy made visible -- the camera')
print('can sit further away with a longer lens, or nearer with a shorter one, and')
print('both reproduce the picture. The chains are trapped in whichever basin they')
print('started nearest.')

**Consequence, and it matters for how the 2159 re-fit is read.** More draws
cannot fix a multimodality, and neither can step-size tuning; even HMC is not a
guaranteed remedy for separated modes. It also qualifies the deterministic
re-fit: that fit found *one point* on this ridge, and the posterior says the
ridge is several metres long. 2159's **orientation** remains solidly determined
— the 1.3° agreement with its own compass holds across the run — but its
**distance along the line of sight is uncertain at the several-metre level**,
which the single-point fit did not reveal.

### 1c. The separation is systematic, and it tracks missing feature constraints

2159 is not a special case. Measuring between-chain sd against within-chain sd
for every camera's position and focal length shows **13 of 29 cameras** with
chains sitting in separate regions — and which cameras those are is not random.

In [ ]:
with np.load(WORKSPACE / manifest['state_file']) as s:
    st_keys = [str(k) for k in s['keys']]
    oc = np.asarray(s['obs_cam'], int)
obs = dict(zip(st_keys, np.bincount(oc, minlength=len(st_keys))))

sep = {}
for k in keys:
    bb = keys.index(k) * 7
    sep[k] = max(float(draws[:, :, bb + j].mean(1).std()
                       / max(draws[:, :, bb + j].std(1).mean(), 1e-12)) for j in (0, 1, 2, 6))

print('between-chain sd / within-chain sd  (>1.5 = chains in separate regions)')
print('%7s%9s%10s%9s%9s%10s   %s' % ('cam', 'obs', 'e', 'n', 'u', 'logf', 'worst'))
for k in sorted(keys, key=lambda k: -sep[k]):
    bb = keys.index(k) * 7
    r = [float(draws[:, :, bb + j].mean(1).std() / max(draws[:, :, bb + j].std(1).mean(), 1e-12))
         for j in (0, 1, 2, 6)]
    flag = ' <-- separated' if sep[k] > 1.5 else ''
    print('%7s%9d%10.2f%9.2f%9.2f%10.2f   %5.2f%s' % (k, obs[k], r[0], r[1], r[2], r[3], sep[k], flag))

zero = [k for k in keys if obs[k] == 0]
has = [k for k in keys if obs[k] > 0]
sz = [k for k in zero if sep[k] > 1.5]
sh = [k for k in has if sep[k] > 1.5]
print()
print('cameras with ZERO feature observations : %2d, chain-separated: %2d (%.0f%%), median ratio %.2f'
      % (len(zero), len(sz), 100 * len(sz) / len(zero), np.median([sep[k] for k in zero])))
print('cameras WITH feature observations      : %2d, chain-separated: %2d (%.0f%%), median ratio %.2f'
      % (len(has), len(sh), 100 * len(sh) / len(has), np.median([sep[k] for k in has])))
print()
print('median observations, separated cameras     : %5.0f' % np.median([obs[k] for k in keys if sep[k] > 1.5]))
print('median observations, non-separated cameras : %5.0f' % np.median([obs[k] for k in keys if sep[k] <= 1.5]))

**83% of the cameras with no feature observations have separated chains,
against 18% of those that have them.** Median separation ratio is 2.92 without
observations and 1.01 with them; median observation count is **0** among
separated cameras and **644** among the rest. Of the three separated cameras
that do carry observations, one (2237) has only 17.

So the multimodality is not a quirk of one badly-behaved camera — it is a
systematic failure of the cameras the likelihood does not tie to anything else,
and it is the same set of cameras identified in §3c. That materially raises the
value of adding the missing cross-image constraints: it is not a patch for 2159,
it addresses the mechanism behind 13 cameras at once.

## 2. Where things are in space

Hillshade DEM with the posterior draws plotted, not just point estimates: every
camera's E/N cloud, the antenna cloud, and the transmitter cloud. The clouds are
elongated because the chains move slowly along correlated directions — the
spatial signature of the same stiffness quantified above.
**DEM coordinate convention (corrected 2026-09-22):** this map reads the canonical
`marjum_dem.npz` directly. Under the current `eigsep_terrain`, extended caches
already carry their E/N offsets; passing one through legacy `working_grid` shifts
it a second time. The overview and target zoom therefore use no manual offset.


In [ ]:
from eigsep_terrain.marjum_dem import MarjumDEM as DEM
from terrain_hillshade_cmap import plot_terrain_hillshade

# Use the canonical zero-offset DEM. Extended caches already carry their
# offsets in current eigsep_terrain and must not be shifted again here.
dem = DEM(cache_file=str(WORKSPACE / 'marjum_dem.npz'))

ai = names.index('antenna_e')
ti = names.index('transmitter_e') if 'transmitter_e' in names else None
cam_en = np.array([[flat[:, keys.index(k) * 7].mean(), flat[:, keys.index(k) * 7 + 1].mean()]
                   for k in keys])
pts = [cam_en, flat[:, ai:ai + 2].mean(0)[None, :]]
if ti is not None:
    pts.append(flat[:, ti:ti + 2].mean(0)[None, :])
allp = np.vstack(pts)
lo, hi = allp.min(0) - 60, allp.max(0) + 60
e0, n0 = (lo + hi) / 2
rng = float(max(hi - lo) / 2 + 40)
E, N, U = dem.get_tile(erng_m=(e0 - rng, e0 + rng),
                       nrng_m=(n0 - rng, n0 + rng), mesh=False)

fig, ax = plt.subplots(figsize=(10.5, 9.5))
im = plot_terrain_hillshade(ax, U, res=float(dem.res), extent=(E[0], E[-1], N[0], N[-1]))
fig.colorbar(im, ax=ax, label='Elevation [m]')
TH = max(1, (nchain * ndraw) // 4000)
colors = plt.cm.tab20(np.linspace(0, 1, len(keys)))
for k, c in zip(keys, colors):
    b = keys.index(k) * 7
    ax.scatter(flat[::TH, b], flat[::TH, b + 1], s=1.4, color=c, alpha=.35,
               linewidths=0, rasterized=True)
    ax.annotate(k, (flat[:, b].mean(), flat[:, b + 1].mean()), fontsize=6,
                xytext=(3, 3), textcoords='offset points')
ax.scatter(flat[::TH, ai], flat[::TH, ai + 1], s=3, color='k', alpha=.5,
           linewidths=0, rasterized=True, label='antenna draws')
if ti is not None:
    ax.scatter(flat[::TH, ti], flat[::TH, ti + 1], s=3, color='red', alpha=.5,
               linewidths=0, rasterized=True, label='transmitter draws')
shared = json.loads((WORKSPACE / 'v0001_marjum_geometry_snapshot' / 'shared.json').read_text())
rec = shared['transmitter']['recommended_for_propagation']
#ax.plot(*rec['position_enu_m'][:2], 'cX', ms=12, mec='k', label=f"transmitter bracket (±{rec['bound_m']} m)")
ax.add_patch(Circle(rec['position_enu_m'][:2], rec['bound_m'], fill=False, color='c', ls='--'))
ax.set_xlabel('East [m]'); ax.set_ylabel('North [m]'); ax.set_aspect('equal')
ax.set_title('Posterior draws: every camera, the antenna and the transmitter')
ax.legend(loc='best', fontsize=8)
plt.tight_layout(); plt.show()

## 2b. Zoomed: the two targets on the hillshade

The map above is dominated by the 29-camera network, and the two targets are a
single smudge inside it. Here they are on their own, at a scale where the draw
clouds are actually resolved. Left is 240 m of context with the zoom box marked;
right is the zoom. **No point-estimate markers** — the clouds are the raw draws.

The antenna and the transmitter are only ~6.6 m apart in E/N but ~94 m apart
vertically, so in plan view this is a near-vertical pair: the map barely
separates them, which is part of why the vertical disagreement has been the hard
part of this geometry.

In [ ]:
from matplotlib.patches import Rectangle

tgt = np.vstack([flat[:, ai:ai + 2], flat[:, ti:ti + 2]]) if ti is not None else flat[:, ai:ai + 2]
ctr = (tgt.min(0) + tgt.max(0)) / 2
half = float(max(tgt.max(0) - tgt.min(0)) / 2) + 3.0
TZ = max(1, (nchain * ndraw) // 20000)

fig, axs = plt.subplots(1, 2, figsize=(11.5, 5.4))
for ax, rad, ttl in ((axs[0], 120., 'context - 240 m across'),
                     (axs[1], half, 'zoom - %.0f m across' % (2 * half))):
    Ez, Nz, Uz = dem.get_tile(erng_m=(ctr[0] - rad, ctr[0] + rad),
                              nrng_m=(ctr[1] - rad, ctr[1] + rad), mesh=False)
    plot_terrain_hillshade(ax, Uz, res=float(dem.res),
                           extent=(Ez[0], Ez[-1], Nz[0], Nz[-1]))
    ax.scatter(flat[::TZ, ai], flat[::TZ, ai + 1], s=4, color='k', alpha=.45,
               linewidths=0, rasterized=True, label='antenna draws')
    if ti is not None:
        ax.scatter(flat[::TZ, ti], flat[::TZ, ti + 1], s=4, color='red', alpha=.45,
                   linewidths=0, rasterized=True, label='transmitter draws')
    ax.set_aspect('equal'); ax.tick_params(labelsize=7)
    ax.set_xlabel('East [m]', fontsize=8); ax.set_ylabel('North [m]', fontsize=8)
    ax.set_title(ttl, fontsize=9)
axs[0].add_patch(Rectangle((ctr[0] - half, ctr[1] - half), 2 * half, 2 * half,
                           fill=False, color='yellow', lw=1.2))
axs[1].legend(fontsize=7, loc='upper right', framealpha=.85)
fig.suptitle('Antenna and transmitter posterior draws - no point-estimate markers', fontsize=10)
plt.tight_layout(); plt.show()

print('E/N draw sd  antenna     %.3f E  %.3f N  m' % (flat[:, ai].std(), flat[:, ai + 1].std()))
if ti is not None:
    print('E/N draw sd  transmitter %.3f E  %.3f N  m' % (flat[:, ti].std(), flat[:, ti + 1].std()))
    print('E/N separation of the two cloud centres: %.2f m'
          % np.hypot(*(flat[:, ai:ai + 2].mean(0) - flat[:, ti:ti + 2].mean(0))))

## 3. Horizon spread across the posterior, all cameras

How much the *predicted horizon* moves across posterior draws, per camera. This
converts the abstract mixing statistics into the observable they affect. Each
camera's panel is in the companion per-camera document; the table ranks them.

In [ ]:
if hr_summary:
    rows = sorted(hr_summary.items(), key=lambda kv: -(kv[1]['median_horizon_spread_px'] or 0))
    print(f"{'camera':>8}{'median spread px':>18}{'max spread px':>15}{'logf ESS':>10}{'logf Rhat':>11}")
    print('-' * 62)
    for k, v in rows:
        print(f"{k:>8}{v['median_horizon_spread_px']:18.0f}{(v['max_horizon_spread_px'] or 0):15.0f}"
              f"{v['logf_ess']:10.0f}{v['logf_rhat']:11.2f}")
    sp = np.array([v['median_horizon_spread_px'] for _, v in rows], float)
    es = np.array([v['logf_ess'] for _, v in rows], float)
    ok = np.isfinite(sp) & np.isfinite(es)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(es[ok], sp[ok], s=28)
    for (k, v), x, y in zip(rows, es, sp):
        if np.isfinite(x) and np.isfinite(y):
            ax.annotate(k, (x, y), fontsize=6, xytext=(3, 2), textcoords='offset points')
    ax.set_xscale('log'); ax.set_xlabel('focal-length bulk ESS (log)')
    ax.set_ylabel('median horizon spread across draws (px)')
    ax.set_title('Stiffer focal length -> wider predicted-horizon spread', fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print('horizon_range/summary.json not present; run marjum_horizon_range.py first')

## 3b. Posterior scatter, zoomed — raw draws only

The two targets at their own scale, with **no point-estimate marker overlaid**,
so the shape and spread of the sampled distribution is what you see. Axes are
set from the draws themselves. Remember these are draws from a **non-converged**
chain set: read them as where the sampler has been, not as a posterior.

In [ ]:
def scatter_pair(idx, label, color):
    e, n, u = flat[:, idx], flat[:, idx + 1], flat[:, idx + 2]
    fig, axs = plt.subplots(1, 3, figsize=(13, 4.1))
    for ax, (x, y, xl, yl) in zip(axs, [(e, n, 'East [m]', 'North [m]'),
                                        (e, u, 'East [m]', 'Up [m]'),
                                        (n, u, 'North [m]', 'Up [m]')]):
        ax.scatter(x, y, s=1.2, alpha=.18, color=color, linewidths=0, rasterized=True)
        ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.set_xlim(np.percentile(x, .2), np.percentile(x, 99.8))
        ax.set_ylim(np.percentile(y, .2), np.percentile(y, 99.8))
    fig.suptitle(f'{label}: raw posterior draws (sd {e.std():.2f} / {n.std():.2f} / {u.std():.2f} m)',
                 fontsize=10)
    plt.tight_layout(); plt.show()

scatter_pair(names.index('antenna_e'), 'Antenna', 'tab:blue')
if 'transmitter_e' in names:
    scatter_pair(names.index('transmitter_e'), 'Transmitter', 'tab:red')

In [ ]:
# Per-chain colouring of the same draws: separated colours mean the chains
# are sitting in different places rather than sampling one distribution.
for label, key, cmapname in (('Antenna', 'antenna_e', 'viridis'),
                             ('Transmitter', 'transmitter_e', 'plasma')):
    if key not in names:
        continue
    i = names.index(key)
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.1))
    cols = plt.get_cmap(cmapname)(np.linspace(0, .9, nchain))
    for c in range(nchain):
        axs[0].scatter(draws[c, :, i], draws[c, :, i + 1], s=1.2, alpha=.25,
                       color=cols[c], linewidths=0, rasterized=True)
        axs[1].scatter(draws[c, :, i], draws[c, :, i + 2], s=1.2, alpha=.25,
                       color=cols[c], linewidths=0, rasterized=True)
    axs[0].set_xlabel('East [m]', fontsize=8); axs[0].set_ylabel('North [m]', fontsize=8)
    axs[1].set_xlabel('East [m]', fontsize=8); axs[1].set_ylabel('Up [m]', fontsize=8)
    for ax in axs:
        ax.tick_params(labelsize=7)
    fig.suptitle(f'{label}: same draws, coloured by chain (8 chains)', fontsize=10)
    plt.tight_layout(); plt.show()

### 3b-ii. Corner plot — all six target coordinates together

The panels above show each target on its own. This shows the **six ENU
coordinates jointly**: diagonal panels are the 1D marginals, off-diagonal panels
the pairwise 2D marginals, on a common range per coordinate. The bottom-left
block is what the separate plots cannot show — the **antenna-transmitter cross
terms**, which are non-zero because both targets are pinned by the same camera
network and inherit its errors together.

Two things to read off it, both caveated by the run not having converged:

- the antenna block is smooth and close to Gaussian, while every panel
  involving `transmitter_E` shows **horizontal striping**. That striping is the
  random walk sticking — long runs of rejected proposals leaving the coordinate
  at one value — not structure in the distribution;
- the strongest cross-target term is `antenna_N`-`transmitter_N`. A correlated
  pair means the two positions cannot be given independent error bars from this
  run; the *difference* between them is better determined than either alone.

In [ ]:
if 'transmitter_e' in names:
    LBL = ['antenna E', 'antenna N', 'antenna U',
           'transmitter E', 'transmitter N', 'transmitter U']
    X = flat[:, [ai, ai + 1, ai + 2, ti, ti + 1, ti + 2]]
    ncorner = 6
    # A common range per coordinate so each diagonal shares the axis of its column.
    rngs = [(np.percentile(X[:, k], .1), np.percentile(X[:, k], 99.9)) for k in range(ncorner)]
    fig, axs = plt.subplots(ncorner, ncorner, figsize=(10.2, 10.2))
    for i in range(ncorner):
        for j in range(ncorner):
            ax = axs[i, j]
            if j > i:
                ax.axis('off')
                continue
            if i == j:
                ax.hist(X[:, i], bins=70, range=rngs[i], color='0.35', histtype='stepfilled')
                ax.set_yticks([])
            else:
                ax.hist2d(X[:, j], X[:, i], bins=70, cmap='magma',
                          range=[rngs[j], rngs[i]], rasterized=True)
                ax.set_ylim(*rngs[i])
            ax.set_xlim(*rngs[j])
            ax.ticklabel_format(useOffset=False, style='plain')
            ax.tick_params(labelsize=6)
            if i < ncorner - 1:
                ax.set_xticklabels([])
            else:
                ax.set_xlabel(LBL[j] + ' [m]', fontsize=7)
                for t in ax.get_xticklabels():
                    t.set_rotation(45)
                    t.set_ha('right')
            if j == 0 and i > 0:
                ax.set_ylabel(LBL[i] + ' [m]', fontsize=7)
            else:
                ax.set_yticklabels([])
    fig.suptitle('Antenna and transmitter ENU: pairwise posterior marginals '
                 '(%d draws, %d chains pooled)' % (X.shape[0], nchain), fontsize=10)
    plt.tight_layout(rect=[0, 0, 1, .975]); plt.show()

    C = np.corrcoef(X.T)
    # Short labels: the full ones make the row wider than the print page box.
    SHORT = ['ant E', 'ant N', 'ant U', 'tx E', 'tx N', 'tx U']
    print('Posterior correlation matrix  (ant = antenna, tx = transmitter):')
    print(' ' * 8 + ''.join('%9s' % l for l in SHORT))
    for i in range(ncorner):
        print('%-8s' % SHORT[i] + ''.join('%9.3f' % C[i, j] for j in range(ncorner)))
    print()
    print('Largest antenna-transmitter cross term: %.3f  (antenna N - transmitter N)'
          % np.abs(C[:3, 3:]).max())

## 3c. Does the likelihood couple images through shared SIFT features?

**Yes — but not for the cameras that need it most, and that is the explanation
for 2159.**

The joint objective *does* carry tie-point terms. 3,200 landmark 3-D positions
are sampled as free parameters, and each camera's likelihood includes the
reprojection residual of every landmark it observes (11,261 observations,
Student-t with a 4 px scale). Cameras are therefore coupled **through shared
landmark positions** — a well-constrained camera and a poorly-constrained one
that observe the same landmark do pull on each other. There are no explicit
epipolar terms in the sampler; the coupling is via shared structure, which is
the standard bundle-adjustment formulation and is the mechanism being asked
about.

**The catch:** 12 of the 29 cameras contribute **zero** observations, including
every one of the six transmitter-era cameras.

In [ ]:
with np.load(WORKSPACE / manifest['state_file']) as s:
    st_keys = [str(k) for k in s['keys']]
    oc = np.asarray(s['obs_cam'], int)
cnt = np.bincount(oc, minlength=len(st_keys))
zero = [k for k, c in zip(st_keys, cnt) if c == 0]
print(f'{len(oc)} observations over {len(st_keys)} cameras')
print(f'cameras with ZERO feature observations: {len(zero)} -> {zero}')
print()
print('observations per camera:')
for i in np.argsort(cnt):
    flag = '   <-- no cross-image constraint' if cnt[i] == 0 else ''
    print(f'  {st_keys[i]:>6}: {cnt[i]:6d}{flag}')

So for camera **2159** the sampled likelihood reduces to: its own horizon,
its own transmitter-pixel ray, and its own GPS/heading/altitude/focal priors.
**Nothing ties its pose to any other camera.** And both of the terms it does
have are nearly invariant along precisely the degenerate direction — sliding the
camera back while lengthening the lens reproduces both the same skyline and the
same bearing to the transmitter. A ridge with no cross-image constraint across
it is exactly the geometry that produces the two separated modes in §1b.

**Is adding SIFT cross-image constraints plausible? Yes, and the data already
exists.** `cv_features/sift_2159.npz` holds 3,487 keypoints, and the
*deterministic* refit already used a matched 2159↔2199 pair (46 correspondences)
as an epipolar term — it simply never entered the sampler's landmark structure.
Two routes:

1. **Triangulate the existing matches into landmarks** and add them as ordinary
   observations. This fits the current model with no new likelihood terms — the
   machinery is already there and unused for these cameras.
2. **Add explicit epipolar residuals** between camera pairs, as the deterministic
   objective does. Avoids introducing new landmark parameters but adds a term
   type the sampler does not currently have.

Route 1 is the smaller change and directly targets the failure: a triangulated
landmark shared with a well-constrained camera pins depth, which is the one
thing 2159's current terms cannot do. Whether it *resolves* the bimodality is an
empirical question — it would have to be tried — but it attacks the right
degree of freedom, and the fact that 2159 recovered in the deterministic fit
**only** once its feature pair was restored is direct evidence that this
constraint is what pins that camera.

*Design question raised, not acted on — no re-run or reparameterisation here.*

## 4. Read-out

- The run **did not converge** and nothing in it is quotable as a posterior.
- The failure is concentrated, not diffuse: **camera focal lengths** are the
  stiffest class, with `cam2159_logf` the worst coordinate in the run.
- The **only block that mixes is the one sampled exactly** rather than by random
  walk, which is the clearest available evidence that the kernel is the
  bottleneck.
- Chains nevertheless **agree far better than Rhat can certify** — between-chain
  sd of the per-chain means is 0.07–0.30 m (antenna) and 0.11–0.27 m
  (transmitter) — and both posterior means land **inside** the existing
  deterministic brackets.
- The ~1 m vertical transmitter offset from the independent 2210/2211 pair
  **persists here too**, as it did in v4, the 2159 re-fit and the drop control.
- **2159's failure is multimodality, not slow mixing** (§1b), and the reason is
  structural: it carries **no cross-image feature constraint at all** (§3c).

No decision is taken here, and no re-run, reparameterisation or release was
started.